In [ ]:
!pip install -r requirements.txt

In [11]:
import json
import pandas as pd
import numpy as np

from src.utils import *
from src.data_processing import *
from src.chains_data_processing import *
from src.feature_creation import *

In [14]:
match_ids = [
    1886347,
    1899585,
    1925299,
    1953632,
    1996435,    
    2006229,
    2011166,
    2013725,
    2015213,
    2017461,
]

skillcorner_opendata_media_url = "https://media.githubusercontent.com/media/SkillCorner/opendata/master/data/matches/"
skillcorner_opendata_raw_url = "https://raw.githubusercontent.com/SkillCorner/opendata/master/data/matches/"

skillcorner = SkillCornerData()
fc = FeatureCreation()
cdp = GKChains()
utils = Utils()

In [ ]:
td_list = skillcorner.prepare_tracking_data_files(skillcorner_opendata_media_url, match_ids=match_ids)
td_df = skillcorner.load_tracking_data(td_list)
td_df = skillcorner.process_tracking_data_dataframe(td_df)

pm_list = skillcorner.prepare_metadata_files(skillcorner_opendata_raw_url, match_ids=match_ids)
pm_df = skillcorner.load_metadata(pm_list)
pm_df = skillcorner.process_players_metadata_dataframe(pm_df)

de_list = skillcorner.prepare_dynamic_events_files(skillcorner_opendata_raw_url, match_ids=match_ids)
de_df = skillcorner.load_dynamic_events_data(de_list)
de_df = skillcorner.filter_player_possessions(de_df)


In [ ]:
print(f"Total Player Possession events: {len(de_df)}")

gk_possessions_df = de_df[
    de_df['player_position'] == "GK"
].copy()

print(f"GK Player Possession events: {len(gk_possessions_df)}")

In [ ]:
# Step 7: Analyze all possession chains
gk_chains = []
i = 0

for (match_id, chain_id), chain_group in gk_possessions_df.groupby(['match_id', 'possession_chain_id']):
    chain_result = cdp.analyze_gk_chain(chain_group, i)
    i += 1
    if chain_result:
        gk_chains.append(chain_result)

# Convert to DataFrame
gk_chains_df = pd.DataFrame(gk_chains)

In [ ]:
cdp.get_gk_chains_info(gk_chains_df=gk_chains_df)

In [ ]:
synced_df = gk_chains_df.merge(
    td_df,
    left_on=["frame_end"],
    right_on="frame",
    suffixes=("_event", "_tracking")
)

In [ ]:
synced_df.head()

In [ ]:
model_features_df = fc.create_features_dataframe(synced_df.possession_index.unique(), synced_df)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

test_model1 = smf.glm(formula="gk_led_to_final_third ~ d2 + max_x_reached ",
                      data = model_features_df,
                      family = sm.families.Binomial()).fit()

print(test_model1.summary())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(model_features_df[test_model1.params.keys()[1:]], model_features_df['gk_led_to_final_third'], test_size=0.33, random_state=42)

# Train the RandomForest classifier
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

# Predict probabilities for the test set
y_probs = clf.predict_proba(model_features_df[test_model1.params.keys()[1:]])[::, 1]

model_features_df["GKLaunch"] = y_probs

In [ ]:
utils.roc(model_features_df['gk_led_to_final_third'], y_probs=y_probs)

In [ ]:
result_df = model_features_df.groupby(["gkId"])["GKLaunch"].sum().sort_values(ascending=False).reset_index()
result_df = result_df.merge(
    pm_df,
    how="left",
    left_on=["gkId"],
    right_on=["id"]
)

In [ ]:
fc.add_normalized_metrics_to_dataframe(result_df)